# Moon <-> Earth analog scores: SSL encoders + weak supervision + anchor calibration (Colab, T4)

Plan: `docs/ML_PLAN.md`. Specs: `config/categories.json`, `config/experiments.json`, `config/anchors.csv`.

**Run order:** Runtime > Change runtime type > **T4 GPU**, then run top to bottom. Leave `SMALL_RUN = True` for a ~5 min smoke test, then set it to `False` (~45-60 min).

**Data:** put the project folder on Google Drive at `MyDrive/analog_project` (needs `config/`, `data/patches/*.csv`, `data/patches/*_arrays.npz`, `data/catalogs/`), or upload a zip with the optional cell below.

**What is learned and what is not:** SSL learns *which patches look alike* (terrain DEM images, geology text). It does **not** learn *how close is close enough* (e.g. whether 230 K on Earth counts as "like" 100 K on the Moon). That comes from the stated tolerances `tau` in `categories.json` and from the expert anchors. Every learned score is checked against a physics-kernel baseline; if it does not beat it, ship the kernel.

Output: `results/category_scores.parquet` with columns `moon_id, earth_id, category, score` (score in [-1, 1]), plus `results/experiment_recommendations.csv`, `results/ml_eval.json`, `results/embeddings.npz`, `results/calibration.json`.

In [ ]:
# ---- 0. Settings ------------------------------------------------------------------------
SMALL_RUN = True            # quick smoke test: ~500 Moon + ~700 Earth patches, 1 short epoch. False = full run
RUN_SSL_IMAGE = True        # DINOv2-small terrain encoder (needs moon/earth *_arrays.npz)
RUN_SSL_TAB = True          # tabular VICReg encoder on terrain scalars
RUN_TEXT = True             # bge-small geology text encoder + contrastive fine-tune
RUN_ABLATION_TAB = False    # also train tabular encoders for thermal/water (uses ablation_alpha_ssl)
EXPORT_MODE = "targets+topk"  # "targets+topk" (~1-2M rows) or "full" (every Moon x Earth x category, ~100M rows)
TOPK = 25
SEED = 0
ROOT_CANDIDATES = ["/content/drive/MyDrive/analog_project", "/content/analog_project", "."]

import os, sys, json, math, re, time, random, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)
random.seed(SEED); np.random.seed(SEED)
IN_COLAB = "google.colab" in sys.modules


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


if IN_COLAB:
    pip_install("transformers>=4.41", "sentence-transformers>=3.0", "datasets", "accelerate", "pyarrow", "scipy")
    if not Path(ROOT_CANDIDATES[0]).exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:  # noqa: BLE001
            print("Drive not mounted:", e)

ROOT = Path(os.environ["ANALOG_ROOT"]) if os.environ.get("ANALOG_ROOT") else None
if ROOT is None:
    ROOT = next((Path(p) for p in ROOT_CANDIDATES if (Path(p) / "config" / "categories.json").exists()), None)
print("ROOT =", ROOT)

In [ ]:
# ---- 0b. Optional: upload a zip of the project instead of using Drive ------------------
def upload_project_zip(dest="/content/analog_project"):
    from google.colab import files
    import io, zipfile
    up = files.upload()
    for _, data in up.items():
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            z.extractall(dest)
    hits = list(Path(dest).glob("**/config/categories.json"))
    return hits[0].parent.parent if hits else Path(dest)


if ROOT is None and IN_COLAB:
    ROOT = upload_project_zip()
assert ROOT is not None and (ROOT / "config" / "categories.json").exists(), "Set ROOT to the project folder"
RES = Path(os.environ.get("ANALOG_RESULTS", ROOT / "results"))
RES.mkdir(parents=True, exist_ok=True)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    HAS_TORCH = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
except ImportError:
    HAS_TORCH, DEVICE = False, "cpu"
print("torch:", HAS_TORCH, "| device:", DEVICE)
if HAS_TORCH and DEVICE == "cpu" and (RUN_SSL_IMAGE or RUN_TEXT) and not SMALL_RUN:
    print("No GPU: switch the runtime to T4, or keep SMALL_RUN = True.")

## 1. Load config and patch tables
Moon: `moon_patches.csv` (3,600 grid patches + 9 Artemis targets, `is_target=1`). Earth: `earth_candidates.csv` joined with `earth_patches_terrain.csv` and `earth_patches_env.csv` on `patch_id`. Missing Earth tables only make the categories that need them NaN.

In [ ]:
CFG = json.loads((ROOT / "config/categories.json").read_text(encoding="utf-8"))
EXP = json.loads((ROOT / "config/experiments.json").read_text(encoding="utf-8"))
CATS = CFG["categories"]
ANCH = pd.read_csv(ROOT / "config/anchors.csv")
TARGETS = pd.read_csv(ROOT / "config/targets.csv")
SITES = pd.read_csv(ROOT / "config/earth_sites.csv")
PDIR = ROOT / "data/patches"


def scol(df, name):
    """String column or empty strings."""
    return df[name].fillna("").astype(str) if name in df.columns else pd.Series("", index=df.index)


moon = pd.read_csv(PDIR / "moon_patches.csv")
earth = pd.read_csv(PDIR / "earth_candidates.csv")
for fname in ["earth_patches_terrain.csv", "earth_patches_env.csv"]:
    f = PDIR / fname
    if not f.exists():
        print("MISSING (categories that need it will be NaN):", fname)
        continue
    d = pd.read_csv(f).drop_duplicates("patch_id")
    d = d.drop(columns=[c for c in d.columns if c in earth.columns and c != "patch_id"])
    earth = earth.merge(d, on="patch_id", how="left")

if (PDIR / "moon_patches_tmin.csv").exists():          # Diviner GCP diurnal min/max per Moon patch
    d = pd.read_csv(PDIR / "moon_patches_tmin.csv").drop_duplicates("patch_id")
    moon = moon.merge(d.drop(columns=[c for c in d.columns if c in moon.columns and c != "patch_id"]), on="patch_id", how="left")
moon["region"] = "south_pole"
MT = ROOT / "data/ml/moon_tiles.csv"          # merged table: adds the 60S-60N tiles (region == "midlat")
if MT.exists():
    d = pd.read_csv(MT, low_memory=False)
    if "region" in d.columns and d["region"].eq("midlat").any():
        d = d[d["region"].eq("midlat") & ~d["patch_id"].astype(str).isin(moon["patch_id"].astype(str))]
        moon = pd.concat([moon, d[[c for c in d.columns if c in moon.columns]]], ignore_index=True, sort=False)
MZF = ROOT / "data/ml/moon_zones.csv"         # Artemis landing / ice zones: Moon rows for anchors with moon_zone landing / ice
if MZF.exists():
    z = pd.read_csv(MZF)
    if "dvn_tbol_min_k" not in z.columns and "x_temp_min_k" in z.columns:
        z["dvn_tbol_min_k"] = z["x_temp_min_k"]
    z["patch_id"], z["region"], z["is_target"] = z["region_id"].astype(str) + "__" + z["zone"].astype(str), "zone", 0
    moon = pd.concat([moon, z[[c for c in z.columns if c in moon.columns]]], ignore_index=True, sort=False)
moon["is_target"] = pd.to_numeric(moon["is_target"], errors="coerce").fillna(0).astype(int)
moon["patch_id"] = moon["patch_id"].astype(str)
earth["patch_id"] = earth["patch_id"].astype(str)
earth = earth.dropna(subset=["lat", "lon"]).reset_index(drop=True)
moon["lat"], moon["lon"] = moon["center_lat"], moon["center_lon"]
moon["body"], earth["body"] = "moon", "earth"

units_f = ROOT / "data/catalogs/moon_geologic_units.csv"
if units_f.exists():
    U = pd.read_csv(units_f)
    U["unit_text"] = (scol(U, "name") + ". " + scol(U, "unit_class_name") + ". " + scol(U, "description") + " " + scol(U, "interpretation"))
    moon = moon.merge(U[["unit", "unit_text"]].drop_duplicates("unit"), left_on="moon_geo_unit", right_on="unit", how="left").drop(columns=["unit"])
moon["geo_text"] = (scol(moon, "moon_geo_unit_name") + ". " + scol(moon, "moon_geo_unit_age") + ". " + scol(moon, "unit_text")).str.replace(r"\s+", " ", regex=True)

if SMALL_RUN:
    keep = moon["is_target"].eq(1) | moon["region"].eq("zone") | moon.index.isin(moon.sample(min(500, len(moon)), random_state=SEED).index)
    moon = moon[keep].reset_index(drop=True)
    keep = earth["source"].eq("named_site") | earth.index.isin(earth.sample(min(700, len(earth)), random_state=SEED).index)
    earth = earth[keep].reset_index(drop=True)
print(f"{len(moon)} Moon patches ({int(moon['is_target'].sum())} targets) | {len(earth)} Earth patches")
print("Moon rows by region:", moon["region"].value_counts().to_dict())
print(earth["source"].value_counts().to_dict())

## 2. Aliases, text flags, physical features (shared by every model)
`column_aliases` maps the names other scripts may use (e.g. `jaxa_amsr2_soil_moist_mean`) to one canonical name, with unit fixes. `features` turn each body's raw columns into the same physical quantity (e.g. Moon `dvn_frac_ice_within_1m` and Earth `permafrost_frac` -> `ice_1m_frac`). Fallback expressions fill row by row.

In [ ]:
def _stack(xs):
    n = max(np.size(x) for x in xs)
    return np.vstack([np.broadcast_to(np.asarray(x, dtype=float), (n,)) for x in xs])


def nsum(*xs):
    a = _stack(xs)
    out = np.nansum(a, 0)
    out[np.isnan(a).all(0)] = np.nan
    return out


def nmean(*xs):
    with np.errstate(all="ignore"):
        return np.nanmean(_stack(xs), 0)


def contains(s, pat):
    return pd.Series(s).astype(str).str.contains(pat, case=False, regex=True, na=False).to_numpy()


NS = dict(log10=np.log10, log1p=np.log1p, abs=np.abs, clip=np.clip, minimum=np.minimum, maximum=np.maximum,
          where=np.where, exp=np.exp, sqrt=np.sqrt, nan=np.nan, nsum=nsum, nmean=nmean, contains=contains)


def ev(expr, df):
    """Evaluate a config expression (trusted: written by the team) on every row of df -> float array."""
    env = dict(NS)
    env.update({c: df[c] for c in df.columns if isinstance(c, str) and c.isidentifier()})
    out = np.asarray(eval(expr, {"__builtins__": {}}, env))
    if out.ndim == 0:
        out = np.full(len(df), float(out))
    return out.astype(float)


def _unit_fix(s, fix):
    if not fix or s.notna().sum() == 0:
        return s
    mx, med = s.abs().max(), s.abs().median()
    if fix == "percent_to_fraction" and mx > 1.5:
        return s / 100.0
    if fix == "gkg_to_pct" and mx > 100:
        return s / 10.0
    if fix == "permille_to_pct" and mx > 100:
        return s / 10.0
    if fix == "cg_to_g" and med > 10:
        return s / 100.0
    return s


RESOLVED = {}


def resolve_aliases(df):
    orig = list(df.columns)
    num = [c for c in orig if pd.api.types.is_numeric_dtype(df[c])]
    for canon, spec in CFG["column_aliases"].items():
        src = canon if canon in orig else next((c for c in spec.get("candidates", []) if c in orig), None)
        if src is None and spec.get("regex"):
            pool = orig if spec.get("text") else num
            src = next((c for c in pool if re.search(spec["regex"], c, re.I)), None)
        if src is None:
            df[canon] = "" if spec.get("text") else np.nan
        elif spec.get("text"):
            df[canon] = df[src].fillna("").astype(str)
        else:
            df[canon] = _unit_fix(pd.to_numeric(df[src], errors="coerce"), spec.get("unit_fix"))
        RESOLVED[(df["body"].iloc[0], canon)] = src
    return df


def add_flags(df):
    body = df["body"].iloc[0]
    for name, spec in CFG["text_flags"].items():
        if body not in spec["bodies"]:
            df[name] = False
            continue
        blob = pd.Series("", index=df.index)
        for f in spec["fields"]:
            blob = blob + " | " + scol(df, f)
        m = blob.str.contains(spec["regex"], case=False, regex=True, na=False).to_numpy()
        if spec.get("or_expr"):
            m = m | (ev(spec["or_expr"], df) > 0)
        df[name] = m.astype(bool)
    return df


FEATURES = {c: [f["name"] for f in v.get("features", [])] for c, v in CATS.items()}
_all = [n for v in FEATURES.values() for n in v]
assert len(_all) == len(set(_all)), "feature names must be unique across categories"


def add_features(df):
    body = df["body"].iloc[0]
    clash = [n for n in _all if n in df.columns]
    if clash:
        print(f"  [{body}] raw columns overwritten by features of the same name: {clash}")
    for cname, c in CATS.items():
        for f in c.get("features", []):
            exprs = f[body] if isinstance(f[body], list) else [f[body]]
            val = np.full(len(df), np.nan)
            for e in exprs:
                try:
                    v = ev(e, df)
                except Exception as ex:  # noqa: BLE001
                    print(f"  [{body}] {cname}.{f['name']}: '{e}' skipped ({type(ex).__name__}: {ex})")
                    continue
                val = np.where(np.isnan(val), v, val)
            df[f["name"]] = val
    return df


moon = add_features(add_flags(resolve_aliases(moon))).copy()     # .copy() defragments the frame
earth = add_features(add_flags(resolve_aliases(earth))).copy()
print("Earth alias -> source column:")
print({k[1]: v for k, v in RESOLVED.items() if k[0] == "earth"})
cov = pd.DataFrame({b: d[_all].notna().mean() for b, d in (("moon", moon), ("earth", earth))}).round(2)
print("Feature coverage (fraction non-NaN):")
print(cov.T.to_string())

## 3. Spatial blocks (no leakage between neighbouring patches)
Moon: 50 km x 50 km blocks in polar-stereographic x/y. Earth: ~5 deg cells (longitude width widened with latitude). 5 folds; fold 0 is held out for every SSL diagnostic.

In [ ]:
SC = CFG["spatial_cv"]
HOLD = SC["holdout_fold"]


def fold_ids(groups, k, seed):
    u = pd.Series(groups).unique()
    np.random.default_rng(seed).shuffle(u)
    m = {g: i % k for i, g in enumerate(u)}
    return np.array([m[g] for g in groups])


bm = SC["moon_block_km"] * 1000.0
xb, yb = np.floor(moon["x_m"] / bm), np.floor(moon["y_m"] / bm)
lb = bm / 1000.0 / 30.32                        # same block size in degrees of latitude (Moon: 30.3 km / deg)
llb = ("ll" + np.floor(moon["lat"] / lb).astype(int).astype(str) + "_"
       + np.floor((moon["lon"] + 180) / (lb / np.maximum(np.cos(np.radians(moon["lat"])), 0.1))).astype(int).astype(str))
moon["block"] = np.where(xb.notna() & yb.notna(), xb.fillna(0).astype(int).astype(str) + "_" + yb.fillna(0).astype(int).astype(str), llb)
dl = SC["earth_block_deg"]
lon_w = dl / np.maximum(np.cos(np.radians(earth["lat"])), 0.1)
earth["block"] = (np.floor(earth["lat"] / dl).astype(int).astype(str) + "_" + np.floor((earth["lon"] + 180) / lon_w).astype(int).astype(str))
moon["fold"] = fold_ids(moon["block"], SC["folds"], SEED)
earth["fold"] = fold_ids(earth["block"], SC["folds"], SEED)
print("Moon blocks:", moon["block"].nunique(), "| Earth blocks:", earth["block"].nunique())

## 4. Physics kernels (the baseline every learned score must beat)
`kernel = 2*exp(-0.5*mean(((x_M - x_E)/tau)^2)) - 1` in physical units with the stated tolerances `tau`. Geology uses a lithology-class kernel (feldspathic / impactite / basaltic / fines, minus penalties for aqueous sediments, silicic rocks and ice surfaces).

In [ ]:
nM, nE = len(moon), len(earth)


def kernel_matrix(cat, chunk=256):
    feats = [f for f in CATS[cat].get("features", []) if f.get("kernel", True)]
    if not feats:
        return None
    names = [f["name"] for f in feats]
    tau = np.array([f["tau"] for f in feats], dtype=np.float32)
    one = np.array([f.get("one_sided") == "earth_at_least_moon" for f in feats])   # gap = max(0, Moon - Earth)
    A = moon[names].to_numpy(np.float32) / tau
    B = earth[names].to_numpy(np.float32) / tau
    need = max(1, math.ceil(CFG["pair_score"]["min_feature_frac"] * len(names)))
    out = np.full((len(A), len(B)), np.nan, np.float32)
    for i in range(0, len(A), chunk):
        d = A[i:i + chunk, None, :] - B[None, :, :]
        if one.any():
            d[..., one] = np.maximum(d[..., one], 0)
        ok = np.isfinite(d)
        n = ok.sum(-1)
        msq = np.where(ok, d * d, 0).sum(-1) / np.maximum(n, 1)
        k = 2 * np.exp(-0.5 * msq) - 1
        k[n < need] = np.nan
        out[i:i + chunk] = k
    return out


def lithology_kernel():
    g = CATS["geology"]
    cl = g["lithology_classes"]
    CM = np.stack([moon["geo_text"].str.contains(v["moon_regex"], case=False, regex=True, na=False).to_numpy() * v["moon_weight"]
                   for v in cl.values()], 1).astype(np.float32)
    CE = np.stack([earth[v["earth_flag"]].to_numpy(bool) for v in cl.values()], 1).astype(np.float32)
    match = (CM @ CE.T) / np.maximum(CM.sum(1, keepdims=True), 1e-6)
    pen = np.zeros(nE, np.float32)
    for flag, p in g["earth_penalties"].items():
        f = earth[flag].to_numpy(bool)
        if flag == "flag_aqueous_sed":
            f = f & ~earth["flag_impact"].to_numpy(bool)
        pen += p * f
    K = np.clip(match - pen[None, :], -1, 1).astype(np.float32)
    known = (scol(earth, "lith_text").str.len().to_numpy() > 0) | (CE.sum(1) > 0) | (pen > 0)
    K[:, ~known] = np.nan          # no lithology information -> unknown, not a guess
    return K


t0 = time.time()
KERNEL = {c: kernel_matrix(c) for c, v in CATS.items() if v["model"]["type"] not in ("rule", "derived")}
KERNEL["geology"] = lithology_kernel()
HAZARD = kernel_matrix("ops_landing_mobility")
for c, K in KERNEL.items():
    print(f"{c:22s} kernel: finite {np.isfinite(K).mean():.2f}, mean {np.nanmean(K):+.2f}")
print(f"kernels in {time.time() - t0:.1f}s")

## 5. Terrain SSL: DINOv2-small on DEM patches (Moon + Earth, mixed batches)
Channels are computed the same way on both bodies (elevation anomaly, slope, curvature), so the encoder has no body-specific channel. Loss: VICReg on two augmented views + CORAL between the Moon and Earth halves of each batch (pulls the two domains' feature covariances together without forcing one-to-one matches). Trained on folds != holdout.

In [ ]:
IMG = CFG["ssl"]["image"]


def load_terrain_arrays(path, ids):
    p128 = Path(str(path).replace("_arrays.npz", "_arrays_128.npz"))
    if p128.exists():                      # prefer the sharper 128 x 128 (78 m/px) patches
        path = p128
    if not Path(path).exists():
        print("missing", path)
        return np.zeros((len(ids), 3, 64, 64), np.float32), np.zeros(len(ids), bool)
    z = np.load(path, allow_pickle=False)
    NA = int(z["elev"].shape[-1])
    print("terrain arrays:", Path(path).name, f"{NA}x{NA}")
    X = np.zeros((len(ids), 3, NA, NA), np.float32)
    have = np.zeros(len(ids), bool)
    idx = {p: i for i, p in enumerate(z["patch_id"].astype(str))}
    elev_all, slope_all = z["elev"], z["slope"]
    valid_all = z["valid"] if "valid" in z.files else None
    assert elev_all.shape[1] == elev_all.shape[2] and elev_all.shape[1] in (64, 128), f"unexpected array shape {elev_all.shape[1:]}"
    for j, p in enumerate(ids):
        i = idx.get(p)
        if i is None:
            continue
        e, s = elev_all[i].astype(np.float32), slope_all[i].astype(np.float32)
        if valid_all is not None:
            e[valid_all[i] == 0], s[valid_all[i] == 0] = np.nan, np.nan
        if np.isnan(e).mean() > 0.2:
            continue
        e = np.where(np.isnan(e), np.nanmean(e), e)
        s = np.where(np.isnan(s), np.nanmean(s), s)
        e = e - e.mean()
        q = np.pad(e, 1, mode="edge")
        lap = q[:-2, 1:-1] + q[2:, 1:-1] + q[1:-1, :-2] + q[1:-1, 2:] - 4 * e
        X[j, 0] = np.clip(e / 1000.0, -3, 3)
        X[j, 1] = np.clip(s / 30.0, 0, 3)
        X[j, 2] = np.tanh(lap * (NA / 64.0) ** 2 / 50.0)   # curvature, scaled to be resolution-independent
        have[j] = True
    return X, have


XM, haveM = load_terrain_arrays(PDIR / "moon_patch_arrays.npz", moon["patch_id"].tolist())
_mi = np.where(moon["region"].eq("midlat").to_numpy())[0]
if len(_mi) and (PDIR / "moon_midlat_arrays_128.npz").exists():
    _X2, _h2 = load_terrain_arrays(PDIR / "moon_midlat_arrays_128.npz", moon["patch_id"].iloc[_mi].tolist())
    if _X2.shape[-1] == XM.shape[-1]:
        XM[_mi[_h2]], haveM[_mi[_h2]] = _X2[_h2], True
    else:
        print(f"midlat arrays {_X2.shape[-1]} px != polar {XM.shape[-1]} px: midlat tiles left out of image SSL")
    del _X2
elif len(_mi):
    print("no moon_midlat_arrays_128.npz: midlat tiles left out of image SSL (tabular / text still use them)")
XE, haveE = load_terrain_arrays(PDIR / "earth_patch_arrays.npz", earth["patch_id"].tolist())
print(f"arrays: Moon {haveM.mean():.2f}, Earth {haveE.mean():.2f} of patches")

In [ ]:
if HAS_TORCH:
    def to_size(x, S):
        return F.interpolate(x, size=(S, S), mode="bilinear", align_corners=False)

    def dihedral(x):
        B, dev = x.shape[0], x.device
        def m():
            return (torch.rand(B, device=dev) < 0.5)[:, None, None, None]
        x = torch.where(m(), x.flip(3), x)
        x = torch.where(m(), x.flip(2), x)
        return torch.where(m(), x.transpose(2, 3), x)

    def random_resized(x, out, smin=0.6, smax=1.0):
        B, dev = x.shape[0], x.device
        s = torch.empty(B, device=dev).uniform_(smin, smax)
        tx = (torch.rand(B, device=dev) * 2 - 1) * (1 - s)
        ty = (torch.rand(B, device=dev) * 2 - 1) * (1 - s)
        theta = torch.zeros(B, 2, 3, device=dev)
        theta[:, 0, 0], theta[:, 1, 1], theta[:, 0, 2], theta[:, 1, 2] = s, s, tx, ty
        grid = F.affine_grid(theta, (B, x.shape[1], out, out), align_corners=False)
        return F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)

    def augment(x, out):
        x = random_resized(dihedral(x), out)
        gain = torch.empty(x.shape[0], 1, 1, 1, device=x.device).uniform_(0.9, 1.1)
        x = torch.cat([x[:, :1] * gain, x[:, 1:]], 1)
        return x + 0.01 * torch.randn_like(x)

    def vicreg(z1, z2, sim=25.0, var=25.0, cov=1.0):
        z1, z2 = z1.float(), z2.float()
        inv = F.mse_loss(z1, z2)
        def vc(z):
            z = z - z.mean(0)
            std = torch.sqrt(z.var(0) + 1e-4)
            c = (z.T @ z) / max(len(z) - 1, 1)
            off = c - torch.diag(torch.diag(c))
            return F.relu(1 - std).mean(), off.pow(2).sum() / z.shape[1]
        v1, c1 = vc(z1)
        v2, c2 = vc(z2)
        return sim * inv + var * (v1 + v2) / 2 + cov * (c1 + c2)

    def coral(a, b):
        a, b = a.float(), b.float()
        d = a.shape[1]
        return (torch.cov(a.T) - torch.cov(b.T)).pow(2).sum() / (4 * d * d) + (a.mean(0) - b.mean(0)).pow(2).mean()

    class TerrainEncoder(nn.Module):
        def __init__(self, ckpt, train_last=4, proj_dim=1024):
            super().__init__()
            from transformers import Dinov2Model
            self.backbone = Dinov2Model.from_pretrained(ckpt)
            for p in self.backbone.parameters():
                p.requires_grad = False
            for blk in self.backbone.encoder.layer[-train_last:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in self.backbone.layernorm.parameters():
                p.requires_grad = True
            h = self.backbone.config.hidden_size
            self.proj = nn.Sequential(nn.Linear(2 * h, proj_dim), nn.BatchNorm1d(proj_dim), nn.ReLU(inplace=True),
                                      nn.Linear(proj_dim, proj_dim), nn.BatchNorm1d(proj_dim), nn.ReLU(inplace=True),
                                      nn.Linear(proj_dim, proj_dim))

        def represent(self, x):
            t = self.backbone(pixel_values=x).last_hidden_state
            return torch.cat([t[:, 0], t[:, 1:].mean(1)], 1)     # CLS + mean patch token (768-d)

        def forward(self, x):
            r = self.represent(x)
            return r, self.proj(r)


def center_norm(EM, EE):
    """Pooled-mean centring (not per body) + L2 norm. NaN rows stay NaN."""
    mu = np.nanmean(np.vstack([EM, EE]), 0)
    def f(E):
        E = E - mu
        n = np.linalg.norm(E, axis=1, keepdims=True)
        return (E / np.where(n > 0, n, np.nan)).astype(np.float32)
    return f(EM), f(EE)


EMB, LOGS = {}, {}

In [ ]:
def train_terrain_ssl():
    S = IMG["image_size"]
    bs = 32 if SMALL_RUN else 128
    h = bs // 2
    trM = np.where(haveM & (moon["fold"].to_numpy() != HOLD))[0]
    trE = np.where(haveE & (earth["fold"].to_numpy() != HOLD))[0]
    model = TerrainEncoder(IMG["backbone"], IMG["train_last_blocks"], IMG["proj_dim"]).to(DEVICE)
    bb = [p for p in model.backbone.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{"params": bb, "lr": IMG["lr_backbone"]}, {"params": model.proj.parameters(), "lr": IMG["lr_head"]}],
                            weight_decay=IMG["weight_decay"])
    per_epoch = max(1, (len(trM) + len(trE)) // bs)
    total = 20 if SMALL_RUN else IMG["epochs"] * per_epoch
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[IMG["lr_backbone"], IMG["lr_head"]], total_steps=total, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    tM, tE = torch.from_numpy(XM), torch.from_numpy(XE)
    log, t0 = [], time.time()
    model.train()
    for step in range(total):
        x = torch.cat([tM[torch.as_tensor(np.random.choice(trM, h))], tE[torch.as_tensor(np.random.choice(trE, h))]]).to(DEVICE, non_blocking=True)
        v1, v2 = augment(x, S), augment(x, S)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            r1, z1 = model(v1)
            r2, z2 = model(v2)
        loss = vicreg(z1, z2) + IMG["lambda_coral"] * 0.5 * (coral(r1[:h], r1[h:]) + coral(r2[:h], r2[h:]))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        if step % 25 == 0 or step == total - 1:
            log.append((step, float(loss)))
            print(f"step {step + 1}/{total}  loss {loss.item():.3f}  ({time.time() - t0:.0f}s)")
    return model, log


@torch.no_grad() if HAS_TORCH else (lambda f: f)
def embed_images(model, X, have, bs=256, aug=False):
    S = IMG["image_size"]
    model.eval()
    out = np.full((len(X), 2 * model.backbone.config.hidden_size), np.nan, np.float32)
    idx = np.where(have)[0]
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        x = torch.from_numpy(X[j]).to(DEVICE)
        x = augment(x, S) if aug else to_size(x, S)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            out[j] = model.represent(x).float().cpu().numpy()
    return out


TERRAIN_MODEL = None
if RUN_SSL_IMAGE and HAS_TORCH and haveM.any() and haveE.any():
    TERRAIN_MODEL, LOGS["terrain_img"] = train_terrain_ssl()
    rawM, rawE = embed_images(TERRAIN_MODEL, XM, haveM), embed_images(TERRAIN_MODEL, XE, haveE)
    EMB["terrain_img"] = center_norm(rawM, rawE)
    torch.save(TERRAIN_MODEL.state_dict(), RES / "terrain_dinov2_vicreg.pt")
else:
    print("terrain image SSL skipped")

## 6. Tabular SSL (VICReg MLP with SCARF corruption)
Used for terrain scalars (blended with the image embedding). For thermal / water it is an ablation only: with ~6 physical scalars SSL can only re-learn a distance, and it cannot supply tolerances. Inputs are standardised with **pooled** Moon+Earth statistics in physical units (per-body normalisation would match a "warm" 180 K Moon patch to a median Earth site, which is physically wrong).

In [ ]:
TAB = CFG["ssl"]["tabular"]


def tab_inputs(cat):
    feats = FEATURES[cat]
    A, B = moon[feats].to_numpy(np.float32), earth[feats].to_numpy(np.float32)
    allv = np.vstack([A, B])
    mu, sd = np.nanmean(allv, 0), np.nanstd(allv, 0)
    mu[~np.isfinite(mu)] = 0
    sd[~np.isfinite(sd) | (sd == 0)] = 1
    def prep(X):
        Z = (X - mu) / sd
        miss = ~np.isfinite(Z)
        Z[miss] = 0
        return np.hstack([Z, miss.astype(np.float32)]).astype(np.float32)
    return prep(A), prep(B), len(feats)


if HAS_TORCH:
    class TabEncoder(nn.Module):
        def __init__(self, d_in, emb=64):
            super().__init__()
            self.enc = nn.Sequential(nn.Linear(d_in, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, emb))
            self.proj = nn.Sequential(nn.Linear(emb, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 256))

        def forward(self, x):
            e = self.enc(x)
            return e, self.proj(e)

    def scarf(x, k, p=0.3):
        """Replace a share p of the k value columns with the same column from random rows of this (single-body) batch."""
        B = x.shape[0]
        x = x.clone()
        mask = torch.rand(B, k, device=x.device) < p
        perm = torch.randint(0, B, (B,), device=x.device)
        x[:, :k] = torch.where(mask, x[perm, :k], x[:, :k]) + 0.05 * torch.randn(B, k, device=x.device)
        return x

    def train_tab(cat):
        AM, AE, k = tab_inputs(cat)
        trM = np.where(moon["fold"].to_numpy() != HOLD)[0]
        trE = np.where(earth["fold"].to_numpy() != HOLD)[0]
        model = TabEncoder(AM.shape[1]).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=TAB["lr"], weight_decay=1e-4)
        tM, tE = torch.from_numpy(AM).to(DEVICE), torch.from_numpy(AE).to(DEVICE)
        h = TAB["batch"] // 2
        steps = 200 if SMALL_RUN else TAB["steps"]
        model.train()
        for s in range(steps):
            xm = tM[torch.as_tensor(np.random.choice(trM, h), device=DEVICE)]
            xe = tE[torch.as_tensor(np.random.choice(trE, h), device=DEVICE)]
            _, z1 = model(torch.cat([scarf(xm, k), scarf(xe, k)]))
            _, z2 = model(torch.cat([scarf(xm, k), scarf(xe, k)]))
            loss = vicreg(z1, z2)
            opt.zero_grad()
            loss.backward()
            opt.step()
        print(f"{cat}: tabular VICReg final loss {loss.item():.3f}")
        model.eval()
        with torch.no_grad():
            EM, EE = model.enc(tM).cpu().numpy(), model.enc(tE).cpu().numpy()
        return center_norm(EM, EE)


if RUN_SSL_TAB and HAS_TORCH:
    EMB["terrain_tab"] = train_tab("terrain")
if RUN_ABLATION_TAB and HAS_TORCH:
    for c in ("thermal_illumination", "water_ice"):
        EMB[c + "_tab"] = train_tab(c)
print("embeddings:", list(EMB))

## 7. Geology text encoder (bge-small, contrastive fine-tune on ontology-mined triplets)
Moon text = unit name + age + catalog description. Earth text = Macrostrat lithology + description + context (GVP rock type, impact catalog, tags), with country/status fields and body words ("lunar", "Earth" ...) stripped so the encoder cannot tell the body from vocabulary. Triplets: (Moon unit, Earth patch sharing a core lithology class, Earth patch with aqueous-sedimentary or silicic rocks). Zero-shot and fine-tuned embeddings are both kept; the evaluation picks one.

In [ ]:
TXT = CFG["ssl"]["text"]
STRIP = re.compile(r"\b(" + "|".join(map(re.escape, TXT["strip_words"])) + r")\b", re.I)
DROP_KV = re.compile(r"\b(country|status|location|elev_m|last_eruption|diameter_km|age)=[^;|]*[;|]?")


def clean_text(s):
    return re.sub(r"\s+", " ", STRIP.sub(" ", str(s))).strip()[:1000]


moon_text = moon["geo_text"].map(clean_text)
earth_text = (scol(earth, "lith_text") + ". " + scol(earth, "lith_desc") + ". " + scol(earth, "context").map(lambda s: DROP_KV.sub(" ", s))).map(clean_text)


def build_triplets(max_n):
    cl = CATS["geology"]["lithology_classes"]
    neg_pool = earth.index[(earth["flag_aqueous_sed"] & ~earth["flag_impact"])
                           | (earth["flag_silicic"] & ~earth["flag_basaltic"] & ~earth["flag_anorthositic"])].to_numpy()
    rng = np.random.default_rng(SEED)
    ex = []
    for i, r in moon.drop_duplicates("geo_text").iterrows():
        for c in ("feldspathic", "impactite", "basaltic"):
            if not re.search(cl[c]["moon_regex"], r["geo_text"], re.I):
                continue
            pos_pool = earth.index[earth[cl[c]["earth_flag"]]].to_numpy()
            if len(pos_pool) == 0 or len(neg_pool) == 0:
                continue
            for _ in range(4):
                ex.append((moon_text[i], earth_text[rng.choice(pos_pool)], earth_text[rng.choice(neg_pool)]))
    rng.shuffle(ex)
    return ex[:max_n]


def text_embeddings(model_name, finetune):
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader
    st = SentenceTransformer(model_name, device=DEVICE)
    if finetune:
        trip = build_triplets(64 if SMALL_RUN else 3000)
        print(len(trip), "triplets")
        if trip:
            try:
                dl = DataLoader([InputExample(texts=list(t)) for t in trip], shuffle=True, batch_size=TXT["batch"])
                st.fit(train_objectives=[(dl, losses.MultipleNegativesRankingLoss(st))],
                       epochs=1 if SMALL_RUN else TXT["epochs"], warmup_steps=10, show_progress_bar=True)
                st.save(str(RES / "geology_bge_finetuned"))
            except Exception as ex:  # noqa: BLE001
                print("fine-tune failed, keeping zero-shot weights:", ex)
    um, ue = pd.unique(moon_text), pd.unique(earth_text)
    em = st.encode(list(um), batch_size=128, normalize_embeddings=True, convert_to_numpy=True)
    ee = st.encode(list(ue), batch_size=128, normalize_embeddings=True, convert_to_numpy=True)
    mM, mE = {t: k for k, t in enumerate(um)}, {t: k for k, t in enumerate(ue)}
    return center_norm(em[moon_text.map(mM).to_numpy()], ee[earth_text.map(mE).to_numpy()])


if RUN_TEXT and HAS_TORCH:
    try:
        EMB["geology_text_zeroshot"] = text_embeddings(TXT["backbone"], finetune=False)
        EMB["geology_text"] = text_embeddings(TXT["backbone"], finetune=True)
    except Exception as ex:  # noqa: BLE001  (HF download problems -> smaller fallback model)
        print("bge failed:", ex, "-> fallback", TXT["fallback_backbone"])
        EMB["geology_text"] = text_embeddings(TXT["fallback_backbone"], finetune=True)
print("embeddings:", list(EMB))

## 8. Raw pair scores: cosine of category embeddings blended with the physics kernel
`raw = alpha * z(cos) + (1 - alpha) * z(kernel)`, with `z()` standardising over 20k random Moon-Earth pairs. If the kernel is NaN (no data) the pair stays NaN: text or images alone never invent a score.

In [ ]:
RNG = np.random.default_rng(SEED)
RP = (RNG.integers(0, nM, 20000), RNG.integers(0, nE, 20000))


def cos_matrix(EM, EE):
    okM, okE = np.isfinite(EM).all(1), np.isfinite(EE).all(1)
    C = (np.nan_to_num(EM) @ np.nan_to_num(EE).T).astype(np.float32)
    C[~okM, :] = np.nan
    C[:, ~okE] = np.nan
    return C


def zstats(M):
    v = M[RP]
    v = v[np.isfinite(v)]
    return (float(v.mean()), float(v.std()) + 1e-6) if len(v) > 10 else (0.0, 1.0)


def blend(cat, cos=None, alpha=None):
    K = KERNEL[cat]
    km, ks = zstats(K)
    zk = (K - km) / ks
    a = CATS[cat]["model"].get("alpha_ssl", 0.0) if alpha is None else alpha
    if cos is None or a == 0:
        return zk.astype(np.float32)
    cm, cs = zstats(cos)
    zc = (cos - cm) / cs
    return np.where(np.isfinite(zc), a * zc + (1 - a) * zk, zk).astype(np.float32)


def wavg(parts):
    num, den = 0, 0
    for w, M in parts:
        ok = np.isfinite(M)
        num = num + np.where(ok, w * M, 0)
        den = den + w * ok
    return np.where(den > 0, num / np.maximum(den, 1e-9), np.nan).astype(np.float32)


COS = {}
tparts = [(w, cos_matrix(*EMB["terrain_" + k])) for key, w in CATS["terrain"]["model"]["ssl_parts"].items()
          for k in [{"image": "img", "tabular": "tab"}[key]] if "terrain_" + k in EMB]
if tparts:
    COS["terrain"] = wavg(tparts)
if "geology_text" in EMB:
    COS["geology"] = cos_matrix(*EMB["geology_text"])

LEARNABLE = [c for c, v in CATS.items() if v["model"]["type"] not in ("rule", "derived")]
RAW, RAW_K = {}, {}
for c in LEARNABLE:
    RAW_K[c] = blend(c)                                  # kernel-only baseline
    if c in COS:
        RAW[c] = blend(c, COS[c])
    elif c + "_tab" in EMB:
        RAW[c] = blend(c, cos_matrix(*EMB[c + "_tab"]), alpha=CATS[c]["model"].get("ablation_alpha_ssl", 0.3))
    else:
        RAW[c] = RAW_K[c]
    print(f"{c:22s} {'SSL+kernel' if RAW[c] is not RAW_K[c] else 'kernel only'}")

## 9. Weak supervision: physics labeling functions on sampled pairs
Each LF votes +1 / -1 or abstains (missing data abstains). LFs marked `eval_only` are kept out of calibration and used only to check agreement. Weak labels enter calibration with a small total weight (`weak_label_weight_total`); the encoders never see them, so LF agreement stays a fair check.

In [ ]:
mid = {p: i for i, p in enumerate(moon["patch_id"])}
eid = {p: j for j, p in enumerate(earth["patch_id"])}


def nearest(lat, lon, df, R, max_km):
    la1, lo1, la2, lo2 = map(np.radians, (lat, lon, df["lat"].to_numpy(float), df["lon"].to_numpy(float)))
    d = 2 * R * np.arcsin(np.sqrt(np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2))
    k = int(np.nanargmin(d))
    return k if d[k] <= max_km else None


def map_id(x, table, cfg_df, df, R, max_km):
    if x in table:
        return table[x]
    r = cfg_df[cfg_df["id"] == x]
    return nearest(float(r["lat"].iloc[0]), float(r["lon"].iloc[0]), df, R, max_km) if len(r) else None


AP = ANCH.copy()
_zone = AP["moon_zone"].fillna("tile").astype(str) if "moon_zone" in AP.columns else pd.Series("tile", index=AP.index)
AP["i"] = [mid.get(f"{x}__{z}") if z in ("landing", "ice") else map_id(x, mid, TARGETS, moon, 1737.4, 10)
           for x, z in zip(AP["moon_target"], _zone)]
AP["j"] = [map_id(x, eid, SITES, earth, 6371.0, 25) for x in AP["earth_site"]]
print(f"anchors: {len(AP)} rows, {AP['i'].notna().sum()} with a Moon patch, {AP['j'].notna().sum()} with an Earth patch")

n_rand = 5000 if SMALL_RUN else 30000
ii = np.r_[RNG.integers(0, nM, n_rand), AP.dropna(subset=["i", "j"])["i"].astype(int).to_numpy()]
jj = np.r_[RNG.integers(0, nE, n_rand), AP.dropna(subset=["i", "j"])["j"].astype(int).to_numpy()]
tgt = np.where(moon["is_target"].eq(1))[0]
named = np.where(earth["source"].eq("named_site"))[0]
ti, tj = np.meshgrid(tgt, named, indexing="ij")
ii, jj = np.r_[ii, ti.ravel()], np.r_[jj, tj.ravel()]


def pair_frame(ii, jj):
    cm = moon.select_dtypes(include=["number", "bool"]).columns
    ce = earth.select_dtypes(include=["number", "bool"]).columns
    A = moon.iloc[ii][cm].reset_index(drop=True).add_suffix("_M")
    B = earth.iloc[jj][ce].reset_index(drop=True).add_suffix("_E")
    return pd.concat([A, B], axis=1)


PF = pair_frame(ii, jj)


def lf_votes(cat, eval_only):
    lfs = [l for l in CATS[cat].get("labeling_functions", []) if bool(l.get("eval_only", False)) == eval_only]
    V = np.zeros((len(PF), len(lfs)), np.float32)
    for k, l in enumerate(lfs):
        try:
            V[:, k] = l["vote"] * (ev(l["expr"], PF) > 0)
        except Exception as ex:  # noqa: BLE001
            print(f"  LF {l['id']} skipped: {type(ex).__name__}: {ex}")
    return V, [l["id"] for l in lfs]


def weak_label(V):
    n = (V != 0).sum(1)
    return np.where(n > 0, V.sum(1) / np.maximum(n, 1), np.nan)


WEAK = {}
for c in CATS:
    V, ids = lf_votes(c, eval_only=False)
    if not ids:
        continue
    WEAK[c] = weak_label(V)
    print(f"{c:22s} LFs {len(ids)} | coverage {np.isfinite(WEAK[c]).mean():.2f} | "
          f"conflict {(((V > 0).any(1)) & ((V < 0).any(1))).mean():.3f} | mean weak label {np.nanmean(WEAK[c]):+.2f}")

## 10. Calibration to [-1, 1] on expert anchors (+ weak labels), then hard caps
Per category `score = tanh(a * (raw - b))`, `a > 0` (monotone). Prior (used as-is when a category has < 4 anchors): `score = tanh(kernel)`, i.e. the physics tolerances decide; a pair is "sufficient" when its features sit within ~0.75 tau on average. Anchor weight by confidence (high 1.0 / med 0.6 / low 0.3), ridge towards the prior. Leave-one-anchor-out predictions are kept for evaluation. Then `score_cap`, Moon-side `hard_caps` (e.g. PSR-dominated patches cap the thermal score at 0) and Earth-side `earth_caps` (e.g. forest/town/water patches cap terrain and ops).

In [ ]:
from scipy.optimize import minimize
from scipy.stats import spearmanr

CAL = CFG["pair_score"]["calibration"]


def caps_for(cat):
    c = CATS[cat]
    cap = np.full(nM, c.get("score_cap", 1.0), np.float32)
    for hc in c.get("hard_caps", []):
        m = ev(hc["when_moon"], moon) > 0
        cap[m] = np.minimum(cap[m], hc["cap"])
    return cap


def earth_caps_for(cat):
    cap = np.ones(nE, np.float32)
    for hc in CATS[cat].get("earth_caps", []):
        m = ev(hc["when_earth"], earth) > 0
        cap[m] = np.minimum(cap[m], hc["cap"])
    return cap


def fit_tanh(ra, ya, wa, rw, yw, prior):
    a0, b0, span = prior
    ok = np.isfinite(ra) & np.isfinite(ya)
    if ok.sum() < CAL["min_anchors_to_fit"]:
        return dict(a=a0, b=b0, fitted=False, n_anchor=int(ok.sum()))
    ra, ya, wa = ra[ok], ya[ok], wa[ok]
    okw = np.isfinite(rw) & np.isfinite(yw)
    rw, yw = rw[okw], yw[okw]
    lam = CAL["weak_label_weight_total"] * wa.sum()
    def loss(p):
        a, b = math.exp(p[0]), p[1]
        L = np.sum(wa * (np.tanh(a * (ra - b)) - ya) ** 2)
        if len(rw):
            L += lam * np.mean((np.tanh(a * (rw - b)) - yw) ** 2)
        return L + CAL["ridge_to_prior"] * ((p[0] - math.log(a0)) ** 2 + ((b - b0) / span) ** 2)
    r = minimize(loss, [math.log(a0), b0], method="Nelder-Mead")
    return dict(a=math.exp(r.x[0]), b=float(r.x[1]), fitted=True, n_anchor=int(ok.sum()))


def kernel_prior(cat):
    """No anchors -> score = tanh(kernel): raw is in kernel z-units, so a0 = std_k and b0 = -mean_k/std_k."""
    km, ks = zstats(KERNEL[cat])
    return (ks, -km / ks, 1.0 / ks)


def calibrate(cat, R, prior=None):
    A = AP[(AP["category"] == cat)].dropna(subset=["i", "j"])
    ai, aj = A["i"].astype(int).to_numpy(), A["j"].astype(int).to_numpy()
    ra, ya = R[ai, aj], A["label"].to_numpy(float)
    wa = A["confidence"].map(CAL["anchor_weight"]).fillna(0.3).to_numpy(float)
    rw = R[ii, jj] if cat in WEAK else np.array([])
    yw = WEAK.get(cat, np.array([]))
    prior = prior or kernel_prior(cat)
    p = fit_tanh(ra, ya, wa, rw, yw, prior)
    cap, capE = caps_for(cat), earth_caps_for(cat)
    loo = np.full(len(A), np.nan)
    for k in range(len(A)):                      # leave-one-anchor-out
        m = np.arange(len(A)) != k
        q = fit_tanh(ra[m], ya[m], wa[m], rw, yw, prior)
        loo[k] = min(np.tanh(q["a"] * (ra[k] - q["b"])), cap[ai[k]], capE[aj[k]])
    S = np.minimum(np.minimum(np.tanh(p["a"] * (R - p["b"])), cap[:, None]), capE[None, :]).astype(np.float32)
    return S, p, pd.DataFrame(dict(moon_id=A["moon_target"].values, earth_id=A["earth_site"].values, label=ya, loo_pred=loo))


SCORES, PARAMS, LOO = {}, {}, {}
for c in LEARNABLE:
    SCORES[c], PARAMS[c], LOO[c] = calibrate(c, RAW[c])
    print(f"{c:22s} a={PARAMS[c]['a']:.3f} b={PARAMS[c]['b']:+.3f} fitted={PARAMS[c]['fitted']} anchors={PARAMS[c]['n_anchor']}")

# ---- derived: landing / mobility / operations
w = CATS["ops_landing_mobility"]["model"]["weights"]
parts = [(w["terrain"], SCORES["terrain"]), (w["dust_regolith"], SCORES["dust_regolith"]),
         (w["thermal_illumination"], SCORES["thermal_illumination"]), (w["hazard_kernel"], HAZARD)]
raw_ops = wavg(parts)
raw_ops[~(np.isfinite(SCORES["terrain"]) | np.isfinite(HAZARD))] = np.nan       # no terrain info -> no ops score
SCORES["ops_landing_mobility"], PARAMS["ops_landing_mobility"], LOO["ops_landing_mobility"] = calibrate(
    "ops_landing_mobility", raw_ops, prior=(1.0, 0.0, 1.0))

# ---- caves: rule only, for the pit target (not in the south-pole grid)
cv = CATS["caves_subsurface"]
CAVE = np.full(nE, cv["default_score"], np.float32)
done = np.zeros(nE, bool)
for r in cv["rules_first_match"]:
    m = (ev(r["expr"], earth) > 0) & ~done
    CAVE[m], done = r["score"], done | m
CAVE = np.minimum(CAVE, cv["score_cap"])
print("caves rule: Earth patches scoring >= 0.5:", int((CAVE >= 0.5).sum()))

## 11. Label-free evaluation (+ anchor checks)
1. **Augmentation invariance** (terrain, held-out blocks): cos(x, aug(x)) should be >= 0.9 and far above random pairs.
2. **Domain probe AUC** (held-out blocks): Moon vs Earth from the embedding. ~0.5-0.8 good; > 0.9 = the embedding mostly encodes the body -> lower `alpha_ssl` or ship the kernel.
3. **Known-analog retrieval**: percentile of each positive anchor (label >= 0.5) among all Earth patches for its Moon target (want >= 90th); negative anchors (<= -0.5) should sit low.
4. **Physically impossible pairs**: share scoring >= 0 (want < 5%) and >= 0.5 (want < 1%).
5. **Held-out LF agreement** (`eval_only` LFs): sign agreement.
6. **Anchor leave-one-out**: MAE and Spearman. Small n: read as a sanity check, not a benchmark.
7. **Baseline**: Spearman of raw scores vs anchors, kernel-only vs SSL blend. Ship SSL only where it wins.

In [ ]:
def auc(score, y):
    r = pd.Series(score).rank().to_numpy()
    n1, n0 = int(y.sum()), int(len(y) - y.sum())
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / max(n1 * n0, 1))


def domain_auc(EM, EE):
    X = np.vstack([EM, EE])
    y = np.r_[np.ones(len(EM)), np.zeros(len(EE))]
    f = np.r_[moon["fold"].to_numpy(), earth["fold"].to_numpy()]
    ok = np.isfinite(X).all(1)
    X, y, f = X[ok], y[ok], f[ok]
    Xb = np.hstack([X, np.ones((len(X), 1))])
    tr, te = f != HOLD, f == HOLD
    if tr.sum() < 10 or te.sum() < 10 or len(np.unique(y[te])) < 2:
        return np.nan
    wv = np.linalg.solve(Xb[tr].T @ Xb[tr] + 1.0 * np.eye(Xb.shape[1]), Xb[tr].T @ (2 * y[tr] - 1))
    return auc(Xb[te] @ wv, y[te].astype(int))


EVAL = {"diagnostics": {}, "categories": {}}
for k, (EM, EE) in EMB.items():
    EVAL["diagnostics"][f"domain_auc_{k}"] = domain_auc(EM, EE)

if TERRAIN_MODEL is not None:
    hm = np.where(haveM & (moon["fold"].to_numpy() == HOLD))[0][:256]
    he = np.where(haveE & (earth["fold"].to_numpy() == HOLD))[0][:256]
    Xh = np.concatenate([XM[hm], XE[he]])
    hv = np.ones(len(Xh), bool)
    e0, e1 = embed_images(TERRAIN_MODEL, Xh, hv), embed_images(TERRAIN_MODEL, Xh, hv, aug=True)
    n0 = e0 / np.linalg.norm(e0, axis=1, keepdims=True)
    n1 = e1 / np.linalg.norm(e1, axis=1, keepdims=True)
    EVAL["diagnostics"]["terrain_aug_invariance_cos"] = float((n0 * n1).sum(1).mean())
    EVAL["diagnostics"]["terrain_random_pair_cos"] = float((n0 * n0[np.random.permutation(len(n0))]).sum(1).mean())


def retrieval(cat, S):
    A = AP[AP["category"] == cat].dropna(subset=["i", "j"])
    pos, neg = [], []
    for _, r in A.iterrows():
        row = S[int(r["i"])]
        v = row[int(r["j"])]
        if not np.isfinite(v):
            continue
        pct = float((row[np.isfinite(row)] < v).mean())
        (pos if r["label"] >= 0.5 else neg if r["label"] <= -0.5 else []).append(pct)
    return pos, neg


for c, S in SCORES.items():
    out = {}
    pos, neg = retrieval(c, S)
    out["pos_anchor_percentile_median"] = float(np.median(pos)) if pos else None
    out["neg_anchor_percentile_median"] = float(np.median(neg)) if neg else None
    sv = S[ii, jj]
    for ip in CATS[c].get("impossible_pairs", []):
        try:
            m = (ev(ip["expr"], PF) > 0) & np.isfinite(sv)
        except Exception as ex:  # noqa: BLE001
            print(c, ip["id"], "skipped:", ex)
            continue
        if m.sum():
            out[f"impossible_{ip['id']}_n"] = int(m.sum())
            out[f"impossible_{ip['id']}_ge0"] = float((sv[m] >= 0).mean())
            out[f"impossible_{ip['id']}_ge05"] = float((sv[m] >= 0.5).mean())
    V, ids = lf_votes(c, eval_only=True)
    for k, lid in enumerate(ids):
        hit = (V[:, k] != 0) & np.isfinite(sv)
        if hit.sum():
            out[f"lf_agree_{lid}"] = float((np.sign(sv[hit]) == np.sign(V[hit, k])).mean())
            out[f"lf_agree_{lid}_n"] = int(hit.sum())
    L = LOO[c].dropna()
    if len(L) >= 3:
        out["anchor_loo_mae"] = float(np.abs(L["loo_pred"] - L["label"]).mean())
        out["anchor_loo_spearman"] = float(spearmanr(L["loo_pred"], L["label"]).correlation)
    if c in RAW_K:
        A = AP[AP["category"] == c].dropna(subset=["i", "j"])
        ai, aj = A["i"].astype(int).to_numpy(), A["j"].astype(int).to_numpy()
        variants = [("kernel", RAW_K[c]), ("blend", RAW[c])]
        if c == "geology" and "geology_text_zeroshot" in EMB:      # zero-shot vs fine-tuned text encoder
            variants.append(("blend_zeroshot", blend("geology", cos_matrix(*EMB["geology_text_zeroshot"]))))
        for nm, R in variants:
            v = R[ai, aj]
            ok = np.isfinite(v)
            if ok.sum() >= 3:
                out[f"anchor_spearman_raw_{nm}"] = float(spearmanr(v[ok], A["label"].to_numpy()[ok]).correlation)
    out["frac_sufficient"] = float(np.nanmean(S >= 0.5))
    out["frac_avoid"] = float(np.nanmean(S < 0))
    out["frac_nan"] = float(np.isnan(S).mean())
    EVAL["categories"][c] = out

A = AP[AP["category"] == "caves_subsurface"].dropna(subset=["j"])
EVAL["categories"]["caves_subsurface"] = {"rule_vs_anchor": [(r.earth_site, r.label, float(CAVE[int(r.j)])) for r in A.itertuples()]}
print(json.dumps(EVAL, indent=1, default=str)[:6000])

## 12. Export `results/category_scores.parquet` (moon_id, earth_id, category, score)
`targets+topk`: every Artemis target x every Earth patch, every Moon patch x the 30 named sites, and the top-K Earth patches per Moon patch per category, plus the pit x every Earth patch for caves. `full` writes every pair in chunks. Embeddings and calibration are saved too, so any other pair can be scored later.

In [ ]:
mids, eids = moon["patch_id"].to_numpy(), earth["patch_id"].to_numpy()
frames = []
for c, S in SCORES.items():
    if EXPORT_MODE == "full":
        continue
    keep = np.zeros(S.shape, bool)
    keep[tgt, :] = True
    keep[:, named] = True
    Sf = np.where(np.isfinite(S), S, -np.inf)
    k = min(TOPK, nE - 1)
    top = np.argpartition(-Sf, kth=k, axis=1)[:, :k]
    keep[np.arange(nM)[:, None], top] = True
    keep &= np.isfinite(S)
    a, b = np.nonzero(keep)
    frames.append(pd.DataFrame(dict(moon_id=mids[a], earth_id=eids[b], category=c, score=S[a, b])))
pit = cv["applies_to_moon_ids"][0]
frames.append(pd.DataFrame(dict(moon_id=pit, earth_id=eids, category="caves_subsurface", score=CAVE)))
out_path = RES / "category_scores.parquet"
if EXPORT_MODE == "full":
    import pyarrow as pa
    import pyarrow.parquet as pq
    schema = pa.schema([("moon_id", pa.string()), ("earth_id", pa.string()), ("category", pa.string()), ("score", pa.float32())])
    with pq.ParquetWriter(out_path, schema) as wr:
        for c, S in SCORES.items():
            for i0 in range(0, nM, 200):
                blk = S[i0:i0 + 200]
                a, b = np.nonzero(np.isfinite(blk))
                wr.write_table(pa.table(dict(moon_id=mids[i0 + a], earth_id=eids[b], category=np.full(len(a), c),
                                             score=blk[a, b].astype(np.float32)), schema=schema))
        wr.write_table(pa.Table.from_pandas(frames[-1].astype({"score": "float32"}), schema=schema, preserve_index=False))
    print("wrote", out_path)
else:
    OUT = pd.concat(frames, ignore_index=True)
    OUT["score"] = OUT["score"].astype("float32").round(4)
    try:
        OUT.to_parquet(out_path, index=False)
        print("wrote", out_path, len(OUT), "rows")
    except Exception as ex:  # noqa: BLE001  (no pyarrow) -> csv
        OUT.to_csv(RES / "category_scores.csv.gz", index=False)
        print("parquet failed (", ex, ") -> wrote category_scores.csv.gz", len(OUT), "rows")

np.savez_compressed(RES / "embeddings.npz", moon_id=mids.astype(str), earth_id=eids.astype(str),
                    **{f"{k}_moon": v[0] for k, v in EMB.items()}, **{f"{k}_earth": v[1] for k, v in EMB.items()})
(RES / "calibration.json").write_text(json.dumps(PARAMS, indent=1))
(RES / "ml_eval.json").write_text(json.dumps(EVAL, indent=1, default=str))
pd.concat([d.assign(category=c) for c, d in LOO.items()]).to_csv(RES / "anchor_loo.csv", index=False)

## 13. Inference layer: which experiments each Earth site is good for
For every Moon target (incl. the Tranquillitatis pit) x every named Earth site x every experiment in `config/experiments.json`: `sufficient` / `partial` / `avoid` / `not_applicable`, the category scores behind it, and the "also account for" list (experiment caveats + category caveats + gravity, vacuum, radiation and PSR temperatures, always).

In [ ]:
GLOBAL = {g["id"]: g for g in EXP["global_non_reproducible"]}
ALWAYS = [f"{GLOBAL[g]['condition']}: Moon {GLOBAL[g]['moon']} vs Earth {GLOBAL[g]['earth_best']}" for g in EXP["always_include"]]


def decide(x, sc, ok_pre):
    req = x["required"]
    vals = {c: sc.get(c, np.nan) for c in req}
    if not ok_pre or any(not np.isfinite(v) for v in vals.values()):
        return "not_applicable", vals
    if any(v < x.get("avoid_below", EXP["default_avoid_below"]) for v in vals.values()):
        return "avoid", vals
    if all(vals[c] >= m for c, m in req.items()):
        return "sufficient", vals
    return "partial", vals


rows = []
for _, t in TARGETS[TARGETS["body"] == "moon"].iterrows():
    i = mid.get(t["id"])
    mrow = (moon.iloc[[i]] if i is not None else pd.DataFrame([{"patch_id": t["id"]}])).assign(tags=t["tags"]).reset_index(drop=True)
    for j in named:
        sc = {c: float(S[i, j]) for c, S in SCORES.items()} if i is not None else {}
        if t["id"] in cv["applies_to_moon_ids"]:
            sc["caves_subsurface"] = float(CAVE[j])
        for x in EXP["experiments"]:
            ok_pre = True
            if x.get("moon_precondition"):
                try:
                    ok_pre = bool(ev(x["moon_precondition"], mrow)[0] > 0)
                except Exception:  # noqa: BLE001  (column absent for this Moon target)
                    ok_pre = False
            dec, vals = decide(x, sc, ok_pre)
            weak = [c for c, m in x.get("supporting", {}).items() if np.isfinite(sc.get(c, np.nan)) and sc[c] < m]
            fails = [c for c, m in x["required"].items() if np.isfinite(vals[c]) and vals[c] < m]
            cav = x["also_account_for"] + [cv_ for c in x["required"] for cv_ in CATS[c]["mandatory_caveats"]] + ALWAYS
            rows.append(dict(moon_id=t["id"], earth_id=eids[j], experiment=x["id"], decision=dec,
                             scores="; ".join(f"{c}={v:+.2f}" for c, v in vals.items()),
                             below_minimum=", ".join(fails), weak_supporting=", ".join(weak),
                             also_account_for=" || ".join(dict.fromkeys(cav))))
REC = pd.DataFrame(rows)
REC.to_csv(RES / "experiment_recommendations.csv", index=False)
print(REC["decision"].value_counts().to_dict())


def card(moon_id, earth_id):
    sub = REC[(REC.moon_id == moon_id) & (REC.earth_id == earth_id)]
    print(f"=== {moon_id}  <->  {earth_id}")
    for d in ("sufficient", "partial", "avoid"):
        s = sub[sub.decision == d]
        if len(s):
            print(f"  {d.upper()}: " + "; ".join(f"{r.experiment} [{r.scores}]" for r in s.itertuples()))
    if len(sub):
        print("  ALSO ACCOUNT FOR:")
        for c in sub["also_account_for"].iloc[0].split(" || ")[-len(ALWAYS):]:
            print("   -", c)


card("moon_haworth", "haughton")
card("moon_tranquillitatis_pit", "lava_beds")

## 14. Optional: seed stability
Re-run sections 5-10 with `SEED = 1` and `2`, keep `SCORES` from each run, and compare the top-20 Earth lists per Moon target with `topk_jaccard`. Below ~0.6 the learned ranking is not stable enough to present; use the kernel ranking.

In [ ]:
def topk_jaccard(S1, S2, rows, k=20):
    out = []
    for i in rows:
        a = set(np.argsort(-np.nan_to_num(S1[i], nan=-9))[:k])
        b = set(np.argsort(-np.nan_to_num(S2[i], nan=-9))[:k])
        out.append(len(a & b) / len(a | b))
    return float(np.mean(out))

# example: SCORES_seed0 = {c: S.copy() for c, S in SCORES.items()}; ...re-run with SEED=1...
# print({c: topk_jaccard(SCORES_seed0[c], SCORES[c], tgt) for c in SCORES})